# Validação das métricas do fluxo bruto

Confere se a segmentação (`breaths`) e os eventos inferidos (`respiratory_events`) fazem sentido antes de usá-los para comparar noites.

1. Sinal do fluxo e segmentação
2. Concordância com os eventos da ResMed
3. Perfil médio em torno dos RERAs da ResMed
4. Inspeção visual de exemplos de cada evento
5. Estabilidade por terço da noite
6. Calibração dos limiares de estabilidade e vigília
7. Conferência externa no AirwayLab

Pré-requisito: `uv run sleep-build`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sleep_analysis import respiratory_metrics as rm
from sleep_analysis.breaths import load_flow
from sleep_analysis.db import connect
from sleep_analysis.paths import BREATHS_DIR, CARD_DIR

db = connect()
device_events = db.sql("SELECT * FROM events").df()
inferred_events = db.sql("SELECT * FROM respiratory_events").df()
night_metrics = db.sql("SELECT * FROM night_metrics ORDER BY night").df()


def load_breaths(night: pd.Timestamp) -> pd.DataFrame:
    breaths = pd.read_parquet(BREATHS_DIR / f"{night:%Y%m%d}.parquet")
    return rm.add_baseline(breaths.sort_values("t_start").reset_index(drop=True))


print(f"{len(night_metrics)} noites com fluxo de 25 Hz")
inferred_events.groupby("event_type").size()

## 1. Sinal do fluxo e segmentação

A inspiração deve ser positiva e mais curta que a expiração (Ti/Ttot entre 0,35 e 0,45 em adultos). A frequência pelo fluxo bruto deve bater com o `resp_rate` do aparelho.

In [ ]:
night = night_metrics["night"].iloc[-1]
breaths = load_breaths(night)
flow = load_flow(CARD_DIR / "DATALOG" / f"{night:%Y%m%d}")

regular = breaths[breaths["valid"] & ~breaths["long_pause"]]
duty_cycle = (regular["ti"] / regular["ttot"]).median()
device_rate = db.sql(f"SELECT median(resp_rate) FROM signals_2s WHERE night = '{night:%Y-%m-%d}'").fetchone()[0]
print(f"Ti/Ttot mediano: {duty_cycle:.2f}")
print(f"Frequência pelo fluxo: {regular['rate'].median():.1f} resp/min; aparelho: {device_rate:.1f}")

start = breaths["t_start"].iloc[len(breaths) // 2]
window = flow[(flow["timestamp"] >= start) & (flow["timestamp"] < start + pd.Timedelta(seconds=30))]
marks = breaths[(breaths["t_start"] >= start) & (breaths["t_start"] < start + pd.Timedelta(seconds=30))]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(window["timestamp"], window["flow"], lw=0.8)
for t in marks["t_start"]:
    ax.axvline(t, color="tab:green", lw=0.8)
ax.axhline(0, color="gray", lw=0.5)
ax.set_ylabel("Fluxo (L/s)")
ax.set_title(f"30 s de fluxo em {night:%d/%m} com os inícios de inspiração detectados")

## 2. Concordância com os eventos da ResMed

Os detectores próprios devem enxergar a maior parte do que a ResMed marca (meta: 80% ou mais), e marcar mais eventos, já que são mais sensíveis.

A ResMed marca a apneia no início e a hipopneia no **fim** do evento. Por isso a comparação verifica se a marca da ResMed cai dentro do evento próprio, com folga de 15 s de cada lado.

- Apneias e hipopneias da ResMed contra `hypopnea30`
- RERAs da ResMed (`arousal`) contra `surge` e `rera_like`

In [ ]:
nights_with_flow = set(night_metrics["night"])
device = device_events[device_events["night"].isin(nights_with_flow)]
comparisons = {
    "Apneias/hipopneias ResMed -> hypopnea30": (
        ["obstructive_apnea", "central_apnea", "unclassified_apnea", "hypopnea"],
        "hypopnea30",
    ),
    "RERAs ResMed -> surge": (["arousal"], "surge"),
    "RERAs ResMed -> rera_like": (["arousal"], "rera_like"),
}
MARGIN = np.timedelta64(15, "s")

rows = []
for label, (device_types, inferred_type) in comparisons.items():
    reference = device[device["event_type"].isin(device_types)]
    candidates = inferred_events[inferred_events["event_type"] == inferred_type]
    matched = 0
    for night, group in reference.groupby("night"):
        night_candidates = candidates[candidates["night"] == night]
        starts = night_candidates["timestamp"].to_numpy()
        ends = starts + pd.to_timedelta(night_candidates["duration_s"].fillna(0), unit="s").to_numpy()
        for timestamp in group["timestamp"].to_numpy():
            if ((starts - MARGIN <= timestamp) & (timestamp <= ends + MARGIN)).any():
                matched += 1
    rows.append(
        {
            "comparação": label,
            "eventos ResMed": len(reference),
            "encontrados": matched,
            "recall %": round(matched / len(reference) * 100, 1) if len(reference) else np.nan,
            "eventos próprios": len(candidates),
        }
    )
pd.DataFrame(rows)

## 3. Perfil médio em torno dos RERAs da ResMed

Mediana de cada atributo por respiração, alinhada no RERA marcado pelo aparelho (respiração 0). O padrão esperado é achatamento crescente (`flatness`) e leve queda de volume, seguidos de um salto de ventilação. Esse perfil justificou o critério relativo de `rera_like` (flatness 0,06 acima da base local).

In [ ]:
BEFORE, AFTER = 12, 8
profiles = []
for night, group in device[device["event_type"] == "arousal"].groupby("night"):
    night_breaths = load_breaths(night)
    for timestamp in group["timestamp"]:
        position = night_breaths["t_start"].searchsorted(timestamp)
        if position < BEFORE or position + AFTER > len(night_breaths):
            continue
        window = night_breaths.iloc[position - BEFORE : position + AFTER]
        profiles.append(
            pd.DataFrame(
                {
                    "breath": np.arange(-BEFORE, AFTER),
                    "flatness_delta": (window["flatness"] - window["flatness_baseline"]).to_numpy(),
                    "vt_ratio": window["vt_ratio"].to_numpy(),
                    "vent_ratio": window["vent_ratio"].to_numpy(),
                }
            )
        )
profile = pd.concat(profiles).groupby("breath").median()

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, column in zip(axes, profile.columns):
    ax.plot(profile.index, profile[column], marker="o")
    ax.axvline(0, color="tab:red", linestyle="--")
    ax.set_ylabel(column)
axes[0].set_title(f"Perfil mediano em {len(profiles)} RERAs da ResMed")
axes[-1].set_xlabel("Respiração relativa ao RERA")
fig.tight_layout()

## 4. Inspeção visual

90 s de fluxo em volta de exemplos sorteados de cada tipo de evento. Critério do ApneaBoard: um RERA aparece como inspirações achatadas ou com picos caindo, terminadas por uma respiração abrupta maior. Troque `SEED` para ver outros exemplos.

In [ ]:
SEED = 1
EXAMPLES_PER_TYPE = 4
flow_cache: dict[pd.Timestamp, pd.DataFrame] = {}


def plot_event(ax, night: pd.Timestamp, timestamp: pd.Timestamp, title: str) -> None:
    if night not in flow_cache:
        flow_cache[night] = load_flow(CARD_DIR / "DATALOG" / f"{night:%Y%m%d}")
    night_flow = flow_cache[night]
    begin, end = timestamp - pd.Timedelta(seconds=45), timestamp + pd.Timedelta(seconds=45)
    window = night_flow[(night_flow["timestamp"] >= begin) & (night_flow["timestamp"] < end)]
    ax.plot(window["timestamp"], window["flow"], lw=0.7)
    ax.axvline(timestamp, color="tab:red", linestyle="--")
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])


event_types = ["surge", "hypopnea30", "rera_like", "wake_like"]
fig, axes = plt.subplots(len(event_types), EXAMPLES_PER_TYPE, figsize=(16, 10))
for row, event_type in enumerate(event_types):
    sample = inferred_events[inferred_events["event_type"] == event_type].sample(EXAMPLES_PER_TYPE, random_state=SEED)
    for column, event in enumerate(sample.itertuples()):
        plot_event(axes[row, column], event.night, event.timestamp, f"{event_type} {event.timestamp:%d/%m %H:%M:%S}")
fig.tight_layout()

## 5. Estabilidade por terço da noite

O N3 (sono profundo) se concentra no 1º terço e o REM no último. Se `stable_pct` capta sono profundo, ele deve cair do 1º para o 3º terço.

In [ ]:
by_third = {
    "stable_pct": ["stable_pct_t1", "stable_pct_t2", "stable_pct_t3"],
    "surge_index": ["surge_index_t1", "surge_index_t2", "surge_index_t3"],
    "flow_limited_pct": ["flow_limited_pct_t1", "flow_limited_pct_t2", "flow_limited_pct_t3"],
}
fig, axes = plt.subplots(1, len(by_third), figsize=(14, 4))
for ax, (metric, columns) in zip(axes, by_third.items()):
    ax.boxplot([night_metrics[column].dropna() for column in columns], tick_labels=["1º", "2º", "3º"], showfliers=False)
    ax.set_title(metric)
    ax.set_xlabel("Terço da noite")
fig.tight_layout()
night_metrics[[column for columns in by_third.values() for column in columns]].median().round(2)

## 6. Calibração dos limiares

Distribuição do coeficiente de variação (CV) da duração (`ttot`) e do volume (`vt`) das respirações, nas noites a 7,0 cmH2O, sem as respirações que fazem parte de eventos (senão cada apneia inflaria o CV e viraria "vigília"). As linhas mostram os limiares fixos em `respiratory_metrics.py`: janelas de 5 min abaixo das duas linhas e sem nenhum evento contam como estáveis; janelas de 1 min acima das duas contam como padrão de vigília. Se muitas noites novas entrarem com outra pressão, vale recalibrar.

In [ ]:
pressure_nights = set(db.sql("SELECT night FROM nights WHERE pressure = 7.0").df()["night"])
calibration_nights = [night for night in night_metrics["night"] if night in pressure_nights]

stable_stats, wake_stats = [], []
for night in calibration_nights:
    night_breaths = load_breaths(night)
    rm.mark_events(night_breaths)
    stable_stats.append(rm.window_stats(night_breaths, rm.STABLE_WINDOW, rm.STABLE_MIN_BREATHS))
    wake_stats.append(rm.window_stats(night_breaths, rm.WAKE_WINDOW, rm.WAKE_MIN_BREATHS))
stable_stats, wake_stats = pd.concat(stable_stats), pd.concat(wake_stats)

panels = [
    (stable_stats, "cv_ttot", rm.STABLE_MAX_CV_TTOT, "5 min: CV ttot"),
    (stable_stats, "cv_vt", rm.STABLE_MAX_CV_VT, "5 min: CV vt"),
    (wake_stats, "cv_ttot", rm.WAKE_MIN_CV_TTOT, "1 min: CV ttot"),
    (wake_stats, "cv_vt", rm.WAKE_MIN_CV_VT, "1 min: CV vt"),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (stats, column, threshold, title) in zip(axes, panels):
    ax.hist(stats[column].clip(upper=1), bins=60)
    ax.axvline(threshold, color="tab:red")
    ax.set_title(f"{title} (limiar {threshold}, p{(stats[column] <= threshold).mean() * 100:.0f})")
fig.tight_layout()
print(f"{len(calibration_nights)} noites a 7,0 cmH2O")

## 7. Conferência externa no AirwayLab

Abrir as noites abaixo em [airwaylab.app](https://airwaylab.app) (roda no navegador; carregar a pasta `data/raw/card`) e comparar com a aba de NED:

- `flow_limited_pct` aqui contra *Combined flow limitation %* de lá (mesma definição: NED de 34% ou mais, ou Flatness de 0,85 ou mais)
- `ned_median` e `flatness_median`
- `rera_like_index` contra *RERA-like index* (critérios diferentes: lá é absoluto, aqui é relativo à base local; espere a mesma ordenação entre noites, não os mesmos números)

In [ ]:
recent = night_metrics.tail(30)
picked = pd.concat(
    [
        recent.nsmallest(1, "flow_limited_pct"),
        recent.iloc[[len(recent) // 2]],
        recent.nlargest(1, "flow_limited_pct"),
    ]
)
picked.set_index("night")[["flow_limited_pct", "ned_median", "flatness_median", "rera_like_index", "surge_index"]].round(2)